# 🎲 YAHTZEE - Juega contra la IA

## Cómo jugar:
1. **Lanza los dados** (tienes hasta 3 lanzamientos)
2. **Selecciona qué dados bloquear** para mantenerlos en el siguiente lanzamiento
3. **Elige una categoría** para anotar tus puntos
4. Luego juega la IA (automáticamente)
5. ¡Gana quien tenga más puntos al final!

**Categorías disponibles:**
- **Ones-Sixes:** Suma de esa cara (ej: si tienes 3 unos, anotas 3)
- **Three of a Kind:** Total si hay 3+ dados iguales
- **Four of a Kind:** Total si hay 4+ dados iguales  
- **Full House:** 25 puntos (3 iguales + 2 iguales)
- **Small Straight:** 30 puntos (4 consecutivos: 1-2-3-4, 2-3-4-5, 3-4-5-6)
- **Large Straight:** 40 puntos (5 consecutivos: 1-2-3-4-5 o 2-3-4-5-6)
- **Yahtzee:** 50 puntos (5 dados iguales)
- **Chance:** Total de todos los dados

In [2]:
import random
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ===============================
# Parámetros Montecarlo
# ===============================
SIMS_PER_MASK = 300
SEED = None  # Cambié a None para que sea más aleatorio en el juego interactivo

if SEED is not None:
    random.seed(SEED)

# ===============================
# Utilidades de dados
# ===============================
def roll_single_die() -> int:
    """Devuelve un entero uniforme en [1,6]."""
    return random.randint(1, 6)

def roll_dice(values: List[int], locked: List[bool]) -> None:
    """Lanza dados no bloqueados, actualizando 'values' in place."""
    for i, is_locked in enumerate(locked):
        if not is_locked:
            values[i] = roll_single_die()

def has_sequence(values: List[int], needed_len: int) -> bool:
    """¿Existe una secuencia consecutiva de longitud 'needed_len'?"""
    uniq = sorted(set(values))
    if not uniq:
        return False
    longest = curr = 1
    for i in range(1, len(uniq)):
        if uniq[i] == uniq[i-1] + 1:
            curr += 1
            longest = max(longest, curr)
        else:
            curr = 1
    return longest >= needed_len

# ===============================
# Puntuación
# ===============================
CATEGORIES_ORDER = [
    "ones","twos","threes","fours","fives","sixes",
    "threeOfKind","fourOfKind","fullHouse","smallStraight",
    "largeStraight","yahtzee","chance"
]

CATEGORY_NAMES_ES = {
    "ones": "Unos",
    "twos": "Doses",
    "threes": "Treses",
    "fours": "Cuatros",
    "fives": "Cincos",
    "sixes": "Seises",
    "threeOfKind": "Trío",
    "fourOfKind": "Póker",
    "fullHouse": "Casa Llena",
    "smallStraight": "Escalera Pequeña",
    "largeStraight": "Escalera Grande",
    "yahtzee": "Yahtzee",
    "chance": "Oportunidad"
}

def score_hand(category: str, dice: List[int]) -> int:
    counts = Counter(dice)
    freqs = list(counts.values())
    total = sum(dice)
    if category in ["ones","twos","threes","fours","fives","sixes"]:
        num = {"ones":1, "twos":2, "threes":3, "fours":4, "fives":5, "sixes":6}[category]
        return counts.get(num, 0) * num
    if category == "threeOfKind":
        return total if any(f >= 3 for f in freqs) else 0
    if category == "fourOfKind":
        return total if any(f >= 4 for f in freqs) else 0
    if category == "fullHouse":
        return 25 if (2 in freqs and 3 in freqs) else 0
    if category == "smallStraight":
        return 30 if has_sequence(dice, 4) else 0
    if category == "largeStraight":
        return 40 if has_sequence(dice, 5) else 0
    if category == "yahtzee":
        return 50 if 5 in freqs else 0
    if category == "chance":
        return total
    return 0

def best_category_score(dice: List[int], available: List[str]) -> Tuple[str, int]:
    """Devuelve (mejor_categoria, puntuación) para los dados dados."""
    best_cat, best = None, -1
    for cat in available:
        sc = score_hand(cat, dice)
        if sc > best:
            best = sc
            best_cat = cat
    return best_cat, best

def get_die_emoji(value: int) -> str:
    """Devuelve el emoji del dado según su valor."""
    emojis = {1: "🎲", 2: "🎲", 3: "🎲", 4: "🎲", 5: "🎲", 6: "🎲"}
    return f"{value}"

# ===============================
# Estructuras de juego
# ===============================
@dataclass
class PlayerState:
    name: str
    scores: Dict[str, Optional[int]] = field(default_factory=lambda: {c: None for c in CATEGORIES_ORDER})
    total: int = 0

    def available_categories(self) -> List[str]:
        return [c for c,v in self.scores.items() if v is None]

    def set_score(self, category: str, points: int) -> None:
        self.scores[category] = points
        self.total = sum(v for v in self.scores.values() if v is not None)

@dataclass
class GameState:
    players: List[PlayerState]
    current_player_idx: int = 0
    turn_in_round: int = 0
    dice_values: List[int] = field(default_factory=lambda: [0]*5)
    locked: List[bool] = field(default_factory=lambda: [False]*5)
    rolls_left: int = 3
    game_over: bool = False

    def reset_turn(self):
        self.dice_values = [0]*5
        self.locked = [False]*5
        self.rolls_left = 3

    @property
    def current_player(self) -> PlayerState:
        return self.players[self.current_player_idx]

    def advance_player(self):
        self.current_player_idx = (self.current_player_idx + 1) % len(self.players)
        if self.current_player_idx == 0:
            self.turn_in_round += 1
        if self.turn_in_round >= 13:
            self.game_over = True

# ===============================
# Montecarlo para IA
# ===============================
def all_keep_masks() -> List[List[bool]]:
    """Genera las 32 máscaras posibles de bloqueo para 5 dados."""
    masks = []
    for m in range(32):
        mask = [(m >> i) & 1 == 1 for i in range(5)]
        masks.append(mask)
    return masks

def simulate_expected_value_after_keeps(
    current_values: List[int],
    keep_mask: List[bool],
    rolls_remaining: int,
    available_categories: List[str],
    sims: int
) -> float:
    if rolls_remaining <= 0:
        _, sc = best_category_score(current_values, available_categories)
        return float(sc)

    kept = [v if keep_mask[i] else None for i, v in enumerate(current_values)]
    unlocked_idx = [i for i,b in enumerate(keep_mask) if not b]

    total_score = 0.0
    for _ in range(sims):
        vals = kept[:]
        temp = current_values[:]
        for _r in range(rolls_remaining):
            for i in unlocked_idx:
                temp[i] = roll_single_die()
        for i in range(5):
            vals[i] = vals[i] if vals[i] is not None else temp[i]
        _, sc = best_category_score(vals, available_categories)
        total_score += sc

    return total_score / sims if sims > 0 else 0.0

def choose_keep_mask_montecarlo(
    current_values: List[int],
    rolls_remaining: int,
    available_categories: List[str],
    sims_per_mask: int = SIMS_PER_MASK
) -> List[bool]:
    best_mask = None
    best_ev = -1.0
    for mask in all_keep_masks():
        ev = simulate_expected_value_after_keeps(
            current_values, mask, rolls_remaining - 1, available_categories, sims_per_mask
        )
        if ev > best_ev:
            best_ev = ev
            best_mask = mask
    return best_mask

print("✅ Motor de Yahtzee cargado. Ejecuta la siguiente celda para jugar.")

✅ Motor de Yahtzee cargado. Ejecuta la siguiente celda para jugar.


In [ ]:
# ===============================
# INTERFAZ INTERACTIVA MEJORADA
# ===============================
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import time

class YahtzeeGameUI:
    def __init__(self):
        self.state = GameState(players=[PlayerState("👤 Tú"), PlayerState("🤖 IA")])
        self.game_history = []
        self.statistics = {
            "player_rolls": 0,
            "ia_rolls": 0,
            "max_roll_player": 0,
            "min_roll_player": 30,
            "max_roll_ia": 0,
            "min_roll_ia": 30,
            "categories_used_player": Counter(),
            "categories_used_ia": Counter(),
        }
    
    def print_clear(self, *args, **kwargs):
        """Imprime con saltos de línea."""
        print(*args, **kwargs)
    
    def display_dice_visual(self, values: List[int], locked: List[bool]) -> None:
        """Muestra los dados de forma clara."""
        dice_str = ""
        for i, (val, is_locked) in enumerate(zip(values, locked)):
            lock_icon = "🔒" if is_locked else ""
            dice_str += f"  [{val}]{lock_icon}  "
        
        html = f"""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    padding: 20px; border-radius: 10px; text-align: center; 
                    margin: 15px 0; font-size: 24px; color: white; font-weight: bold;">
            {dice_str}
        </div>
        """
        display(HTML(html))
    
    def display_score_board(self) -> None:
        """Muestra el tablero de puntuación en HTML."""
        board_html = """
        <table style="width: 100%; border-collapse: collapse; background: white; box-shadow: 0 2px 8px rgba(0,0,0,0.1); border-radius: 8px; overflow: hidden;">
        <tr style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); color: white; font-weight: bold;">
            <th style="padding: 12px; text-align: left;">Categoría</th>
            <th style="padding: 12px; text-align: center; width: 100px;">👤 Tú</th>
            <th style="padding: 12px; text-align: center; width: 100px;">🤖 IA</th>
        </tr>
        """
        
        for idx, cat in enumerate(CATEGORIES_ORDER):
            p1_score = self.state.players[0].scores[cat]
            p2_score = self.state.players[1].scores[cat]
            p1_display = f"<b>{p1_score}</b>" if p1_score is not None else "<span style='color: #999;'>-</span>"
            p2_display = f"<b>{p2_score}</b>" if p2_score is not None else "<span style='color: #999;'>-</span>"
            
            row_color = "#f8f9fa" if idx % 2 == 0 else "white"
            board_html += f"""
            <tr style="background: {row_color};">
                <td style="padding: 10px; border-top: 1px solid #eee;">{CATEGORY_NAMES_ES[cat]}</td>
                <td style="padding: 10px; text-align: center; border-top: 1px solid #eee;">{p1_display}</td>
                <td style="padding: 10px; text-align: center; border-top: 1px solid #eee;">{p2_display}</td>
            </tr>
            """
        
        board_html += f"""
        <tr style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); color: white; font-weight: bold;">
            <td style="padding: 12px;">TOTAL</td>
            <td style="padding: 12px; text-align: center; font-size: 18px;">{self.state.players[0].total}</td>
            <td style="padding: 12px; text-align: center; font-size: 18px;">{self.state.players[1].total}</td>
        </tr>
        </table>
        """
        display(HTML(board_html))
    
    def show_available_categories(self) -> None:
        """Muestra categorías disponibles con sus puntuaciones potenciales."""
        available = self.state.players[0].available_categories()
        
        cat_html = "<h4>📊 Categorías disponibles y tus puntos potenciales:</h4><table style='width: 100%; border-collapse: collapse;'>"
        cat_html += "<tr style='background: #f0f0f0; font-weight: bold;'><td style='padding: 8px; border: 1px solid #ddd;'>Categoría</td><td style='padding: 8px; border: 1px solid #ddd;'>Puntos</td></tr>"
        
        for cat in available:
            points = score_hand(cat, self.state.dice_values)
            cat_html += f"<tr><td style='padding: 8px; border: 1px solid #ddd;'>{CATEGORY_NAMES_ES[cat]}</td><td style='padding: 8px; border: 1px solid #ddd; font-weight: bold; color: green;'>{points}</td></tr>"
        
        cat_html += "</table>"
        display(HTML(cat_html))
    
    def show_turn_header(self) -> None:
        """Muestra encabezado del turno."""
        player = self.state.current_player
        header_html = f"""
        <div style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 20px; border-radius: 8px; margin: 20px 0; text-align: center;">
            <h2>TURNO {self.state.turn_in_round + 1}/13</h2>
            <h3>{player.name}</h3>
            <p>Tiradas disponibles: <b>{self.state.rolls_left}</b></p>
        </div>
        """
        display(HTML(header_html))
    
    def _refresh(self, *, show_categories_hint: bool = False) -> None:
        """Redibuja la UI sin acumular salida (evita pantallas gigantes)."""
        clear_output(wait=True)
        self.show_turn_header()
        self.display_dice_visual(self.state.dice_values, self.state.locked)
        self.display_score_board()
        if show_categories_hint:
            self.show_available_categories()

    def play_turn_human(self) -> bool:
        """Turno interactivo del jugador humano."""
        self._refresh(show_categories_hint=True)

        while self.state.rolls_left > 0:
            print(f"\n{'='*60}")
            print("TU TURNO:")
            print("="*60)
            print(f"\n🎲 Tus dados actuales: {self.state.dice_values}")
            print(f"Tiradas disponibles: {self.state.rolls_left}")
            print("\nOpciones:")
            print("  1. Lanzar dados")
            print("  2. Bloquear/desbloquear dados")
            print("  3. Terminar turno y anotar")

            choice = input("\n¿Qué haces? (1/2/3): ").strip()

            if choice == "1":
                roll_dice(self.state.dice_values, self.state.locked)
                self.state.rolls_left -= 1
                self.statistics["player_rolls"] += 1
                total_roll = sum(self.state.dice_values)
                self.statistics["max_roll_player"] = max(self.statistics["max_roll_player"], total_roll)
                self.statistics["min_roll_player"] = min(self.statistics["min_roll_player"], total_roll)
                self._refresh(show_categories_hint=True)

            elif choice == "2":
                print("\nDados actuales:")
                for i, val in enumerate(self.state.dice_values):
                    status = "🔒 BLOQUEADO" if self.state.locked[i] else "🔓 Libre"
                    print(f"  Dado {i+1}: [{val}] {status}")

                die_input = input("\n¿Qué dado bloqueas/desbloqueas? (1-5 o Enter para omitir): ").strip()
                if die_input and die_input.isdigit() and 1 <= int(die_input) <= 5:
                    die_idx = int(die_input) - 1
                    self.state.locked[die_idx] = not self.state.locked[die_idx]
                self._refresh(show_categories_hint=True)

            elif choice == "3":
                break

            else:
                self._refresh(show_categories_hint=True)

        # Elegir categoría (pantalla compacta)
        clear_output(wait=True)
        self.show_turn_header()
        self.display_dice_visual(self.state.dice_values, self.state.locked)
        self.display_score_board()

        print(f"\n{'='*60}")
        print("ELIGE UNA CATEGORÍA PARA ANOTAR:")
        print("="*60)
        available = self.state.players[0].available_categories()

        for i, cat in enumerate(available):
            points = score_hand(cat, self.state.dice_values)
            print(f"  {i+1}. {CATEGORY_NAMES_ES[cat]:20s} → {points} puntos")

        while True:
            cat_choice = input("\n¿Qué categoría? (número): ").strip()
            if cat_choice.isdigit() and 1 <= int(cat_choice) <= len(available):
                chosen_cat = available[int(cat_choice) - 1]
                score = score_hand(chosen_cat, self.state.dice_values)
                self.state.players[0].set_score(chosen_cat, score)
                self.statistics["categories_used_player"][chosen_cat] += 1
                self.game_history.append(f"Jugador anotó {CATEGORY_NAMES_ES[chosen_cat]}: {score}")
                self._refresh(show_categories_hint=False)
                print(f"\n✅ Anotaste {CATEGORY_NAMES_ES[chosen_cat]}: {score} puntos")
                time.sleep(1)
                return True
            print("❌ Opción inválida")
    
    def play_turn_ia(self) -> bool:
        """Turno automático de la IA."""
        player = self.state.players[1]
        self.state.advance_player()
        self.state.reset_turn()

        # Pantalla limpia para que no crezca el output
        clear_output(wait=True)
        print(f"{'='*60}")
        print("🤖 TURNO DE LA IA")
        print("="*60)

        # Tirada inicial
        roll_dice(self.state.dice_values, self.state.locked)
        self.state.rolls_left -= 1
        self.statistics["ia_rolls"] += 1
        total_roll = sum(self.state.dice_values)
        self.statistics["max_roll_ia"] = max(self.statistics["max_roll_ia"], total_roll)
        self.statistics["min_roll_ia"] = min(self.statistics["min_roll_ia"], total_roll)
        print(f"🎲 IA lanzó: {self.state.dice_values}")
        time.sleep(0.8)

        # Rerolls con Montecarlo
        for _ in range(2):
            if self.state.rolls_left <= 0:
                break

            keep_mask = choose_keep_mask_montecarlo(
                self.state.dice_values, self.state.rolls_left, player.available_categories()
            )

            if all(keep_mask):
                print("🤖 IA decide plantarse")
                break

            self.state.locked = keep_mask[:]
            roll_dice(self.state.dice_values, self.state.locked)
            self.state.rolls_left -= 1
            self.statistics["ia_rolls"] += 1
            total_roll = sum(self.state.dice_values)
            self.statistics["max_roll_ia"] = max(self.statistics["max_roll_ia"], total_roll)
            self.statistics["min_roll_ia"] = min(self.statistics["min_roll_ia"], total_roll)
            print(f"🎲 IA relanzó: {self.state.dice_values}")
            time.sleep(0.8)

        # IA elige categoría
        cat, sc = best_category_score(self.state.dice_values, player.available_categories())
        player.set_score(cat, sc)
        self.statistics["categories_used_ia"][cat] += 1
        self.game_history.append(f"IA anotó {CATEGORY_NAMES_ES[cat]}: {sc}")

        # Volver al turno del jugador
        self.state.advance_player()
        self.state.reset_turn()

        # Redibujar UI del jugador (sin acumular)
        self._refresh(show_categories_hint=True)
        print(f"\n✅ IA eligió {CATEGORY_NAMES_ES[cat]}: {sc} puntos")
        time.sleep(1)
        return True
    
    def show_statistics(self) -> None:
        """Muestra estadísticas detalladas del juego."""
        print(f"\n{'='*80}")
        print("📊 ESTADÍSTICAS DETALLADAS DEL JUEGO")
        print("="*80)
        
        stats_html = """
        <div style="background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #667eea;">
        """
        
        # Lanzamientos
        stats_html += "<h3>🎲 Lanzamientos</h3>"
        stats_html += f"<p><b>Tu total de lanzamientos:</b> {self.statistics['player_rolls']}</p>"
        stats_html += f"<p><b>Lanzamiento máximo (total dados):</b> {self.statistics['max_roll_player']}</p>"
        stats_html += f"<p><b>Lanzamiento mínimo (total dados):</b> {self.statistics['min_roll_player']}</p>"
        
        stats_html += "<hr>"
        stats_html += f"<p><b>Lanzamientos de IA:</b> {self.statistics['ia_rolls']}</p>"
        stats_html += f"<p><b>Lanzamiento máximo IA:</b> {self.statistics['max_roll_ia']}</p>"
        stats_html += f"<p><b>Lanzamiento mínimo IA:</b> {self.statistics['min_roll_ia']}</p>"
        
        # Categorías usadas
        stats_html += "<h3>📋 Categorías Utilizadas</h3>"
        stats_html += "<table style='width: 100%; border-collapse: collapse;'>"
        stats_html += "<tr style='background: #667eea; color: white;'><td style='padding: 8px; border: 1px solid #ddd;'><b>Categoría</b></td><td style='padding: 8px; border: 1px solid #ddd;'><b>Jugador</b></td><td style='padding: 8px; border: 1px solid #ddd;'><b>IA</b></td></tr>"
        
        for cat in CATEGORIES_ORDER:
            player_count = self.statistics["categories_used_player"].get(cat, 0)
            ia_count = self.statistics["categories_used_ia"].get(cat, 0)
            if player_count > 0 or ia_count > 0:
                stats_html += f"<tr><td style='padding: 8px; border: 1px solid #ddd;'>{CATEGORY_NAMES_ES[cat]}</td><td style='padding: 8px; border: 1px solid #ddd; text-align: center;'>{player_count}</td><td style='padding: 8px; border: 1px solid #ddd; text-align: center;'>{ia_count}</td></tr>"
        
        stats_html += "</table>"
        stats_html += "</div>"
        display(HTML(stats_html))
    
    def show_game_over(self) -> None:
        """Muestra resultado final (compacto, sin salida gigantesca)."""
        clear_output(wait=True)

        p1_total = self.state.players[0].total
        p2_total = self.state.players[1].total

        if p1_total > p2_total:
            title = "🏆 ¡¡GANASTE!! 🏆"
            bg = "linear-gradient(135deg, #667eea 0%, #764ba2 100%)"
            subtitle = f"{p1_total} - {p2_total}"
        elif p2_total > p1_total:
            title = "😢 Perdiste 😢"
            bg = "linear-gradient(135deg, #f093fb 0%, #f5576c 100%)"
            subtitle = f"IA: {p2_total} vs Tú: {p1_total}"
        else:
            title = "🤝 ¡EMPATE! 🤝"
            bg = "linear-gradient(135deg, #4facfe 0%, #00f2fe 100%)"
            subtitle = f"{p1_total} - {p2_total}"

        header = f"""
        <div style="background: {bg}; color: white; padding: 24px; border-radius: 12px; text-align: center; margin: 12px 0;">
            <h1 style="margin: 0;">{title}</h1>
            <h2 style="margin: 8px 0 0 0;">{subtitle}</h2>
        </div>
        """
        display(HTML(header))

        # Tablero final (ya es una tabla compacta)
        self.display_score_board()

        # Estadísticas en un bloque plegable con scroll
        stats_html = self._build_statistics_html()
        details = f"""
        <details style="margin-top: 14px;">
          <summary style="cursor: pointer; font-weight: 700;">📊 Ver estadísticas detalladas</summary>
          <div style="max-height: 320px; overflow: auto; margin-top: 10px;">{stats_html}</div>
        </details>
        """
        display(HTML(details))

    def _build_statistics_html(self) -> str:
        """Devuelve HTML de estadísticas (sin imprimir cientos de líneas)."""
        parts = []
        parts.append("<div style='background:#f8f9fa; padding: 16px; border-radius: 10px; border-left: 4px solid #667eea;'>")
        parts.append("<h3 style='margin-top:0;'>🎲 Lanzamientos</h3>")
        parts.append(f"<p><b>Tu total de lanzamientos:</b> {self.statistics['player_rolls']}</p>")
        parts.append(f"<p><b>Lanzamiento máximo (suma dados):</b> {self.statistics['max_roll_player']}</p>")
        parts.append(f"<p><b>Lanzamiento mínimo (suma dados):</b> {self.statistics['min_roll_player']}</p>")
        parts.append("<hr>")
        parts.append(f"<p><b>Lanzamientos de IA:</b> {self.statistics['ia_rolls']}</p>")
        parts.append(f"<p><b>Lanzamiento máximo IA:</b> {self.statistics['max_roll_ia']}</p>")
        parts.append(f"<p><b>Lanzamiento mínimo IA:</b> {self.statistics['min_roll_ia']}</p>")

        parts.append("<h3>📋 Categorías Utilizadas</h3>")
        parts.append("<table style='width:100%; border-collapse: collapse;'>")
        parts.append("<tr style='background:#667eea; color:white;'><td style='padding:8px; border:1px solid #ddd;'><b>Categoría</b></td><td style='padding:8px; border:1px solid #ddd; text-align:center;'><b>Jugador</b></td><td style='padding:8px; border:1px solid #ddd; text-align:center;'><b>IA</b></td></tr>")

        for cat in CATEGORIES_ORDER:
            player_count = self.statistics["categories_used_player"].get(cat, 0)
            ia_count = self.statistics["categories_used_ia"].get(cat, 0)
            if player_count > 0 or ia_count > 0:
                parts.append(
                    "<tr>"
                    f"<td style='padding:8px; border:1px solid #ddd;'>{CATEGORY_NAMES_ES[cat]}</td>"
                    f"<td style='padding:8px; border:1px solid #ddd; text-align:center;'>{player_count}</td>"
                    f"<td style='padding:8px; border:1px solid #ddd; text-align:center;'>{ia_count}</td>"
                    "</tr>"
                )

        parts.append("</table>")
        parts.append("</div>")
        return "".join(parts)

    def show_statistics(self) -> None:
        """Compat: ya no imprime; muestra en bloque plegable al final."""
        display(HTML(self._build_statistics_html()))
    
    def run(self) -> None:
        """Ejecuta el juego completo."""
        print("\n" + "="*80)
        print("🎲 BIENVENIDO A YAHTZEE 🎲")
        print("="*80)
        print("\nVas a jugar contra una IA con estrategia inteligente (Montecarlo)")
        print("Tienes 13 rondas para acumular la máxima puntuación.\n")
        time.sleep(2)
        
        while not self.state.game_over:
            # Turno del jugador
            self.play_turn_human()
            
            if self.state.game_over:
                break
            
            # Turno de IA
            self.play_turn_ia()
        
        # Mostrar resultado
        self.show_game_over()

# EJECUTAR EL JUEGO
print("\n🎮 Iniciando Yahtzee interactivo...\n")
game_ui = YahtzeeGameUI()
game_ui.run()



🎮 Iniciando Yahtzee interactivo...


🎲 BIENVENIDO A YAHTZEE 🎲

Vas a jugar contra una IA con estrategia inteligente (Montecarlo)
Tienes 13 rondas para acumular la máxima puntuación.



Categoría,👤 Tú,🤖 IA
Unos,-,-
Doses,-,-
Treses,-,-
Cuatros,-,-
Cincos,-,-
Seises,-,-
Trío,-,-
Póker,-,-
Casa Llena,-,-
Escalera Pequeña,-,-



TU TURNO:

🎲 Tus dados actuales: [0, 0, 0, 0, 0]
Tiradas disponibles: 3

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

🎲 ¡Lanzando dados!
✅ Dados: [6, 4, 2, 2, 4]

TU TURNO:

🎲 Tus dados actuales: [6, 4, 2, 2, 4]
Tiradas disponibles: 2

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

Dados actuales:
  Dado 1: [6] 🔓 Libre
  Dado 2: [4] 🔓 Libre
  Dado 3: [2] 🔓 Libre
  Dado 4: [2] 🔓 Libre
  Dado 5: [4] 🔓 Libre
Dado 3 Bloqueado ✅

TU TURNO:

🎲 Tus dados actuales: [6, 4, 2, 2, 4]
Tiradas disponibles: 2

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

ELIGE UNA CATEGORÍA PARA ANOTAR:
  1. Unos                 → 0 puntos
  2. Doses                → 4 puntos
  3. Treses               → 0 puntos
  4. Cuatros              → 8 puntos
  5. Cincos               → 0 puntos
  6. Seises               → 6 puntos
  7. Trío                 → 0 puntos
  8. Póker                →

Categoría,👤 Tú,🤖 IA
Unos,0,-
Doses,-,-
Treses,-,-
Cuatros,-,-
Cincos,-,-
Seises,-,-
Trío,-,-
Póker,-,-
Casa Llena,-,-
Escalera Pequeña,-,30



TU TURNO:

🎲 Tus dados actuales: [0, 0, 0, 0, 0]
Tiradas disponibles: 3

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

🎲 ¡Lanzando dados!
✅ Dados: [6, 3, 2, 2, 3]

TU TURNO:

🎲 Tus dados actuales: [6, 3, 2, 2, 3]
Tiradas disponibles: 2

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

Dados actuales:
  Dado 1: [6] 🔓 Libre
  Dado 2: [3] 🔓 Libre
  Dado 3: [2] 🔓 Libre
  Dado 4: [2] 🔓 Libre
  Dado 5: [3] 🔓 Libre
Dado 5 Bloqueado ✅

TU TURNO:

🎲 Tus dados actuales: [6, 3, 2, 2, 3]
Tiradas disponibles: 2

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

Dados actuales:
  Dado 1: [6] 🔓 Libre
  Dado 2: [3] 🔓 Libre
  Dado 3: [2] 🔓 Libre
  Dado 4: [2] 🔓 Libre
  Dado 5: [3] 🔒 BLOQUEADO
Dado 3 Bloqueado ✅

TU TURNO:

🎲 Tus dados actuales: [6, 3, 2, 2, 3]
Tiradas disponibles: 2

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar

🎲 ¡Lan

Categoría,👤 Tú,🤖 IA
Unos,0,-
Doses,2,-
Treses,-,-
Cuatros,-,-
Cincos,-,-
Seises,-,-
Trío,-,-
Póker,-,-
Casa Llena,-,-
Escalera Pequeña,-,30



TU TURNO:

🎲 Tus dados actuales: [0, 0, 0, 0, 0]
Tiradas disponibles: 3

Opciones:
  1. Lanzar dados
  2. Bloquear/desbloquear dados
  3. Terminar turno y anotar
